# Análisis de viabilidad del TFM
En `df_fases_MES`tengo la **fase**, **máquina dominante**, y el **grupo/familia de esa máquina**. Con eso puedo montar un  cruce fase × máquina dentro de cada familia, ahi es donde vive la decisión de asignación.

Entonces, la elección real no es "esta fase fue a esta máquina" (eso siempre pasa). La elección existe cuando fases del mismo tipo pudieron ir a máquinas distintas de la misma familia.   
Por eso el análisis interesante es dentro de cada grupo: `¿las fases de una familia se reparten entre sus varias máquinas, o cada tipo de fase va siempre a la misma?`

In [129]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pyarrow as pa

In [130]:
df_FASE_final = pd.read_parquet("res//procesed/df_FASE_final.parquet")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)   
display(df_FASE_final.head(1))

,SmFaseUid,MaquinaDominante,smOrdenId,smMaterialId,smFaseNme,smOrdenNme,MinutosReal_efectivo,MinutosPreparacion_efectivo,MinutosProduccion_efectivo,MinutosReal_total,MinutosPreparacion_total,MinutosProduccion_total,dedicacion_media,n_imputaciones,IDMaquina,NombreMaquina,IDGrupo,CodigoGrupo,NombreGrupo,IDMOTask,IDManufacturingOrder,Description,PreparationTime,ExecutionTime,TimeUnit,ProductEntry,MaterialExit,IDUsualTask,Quantity,ManualStartDate,RealEndDate,MESEntityId
0,00013c31-3895-4370-bb3e-4255371e8d8f,TN09,OF2509545,PE1JNCX0012,TPP,Portacápsulas F704501049704(150713),41.76667,41.76667,0.0,41.76667,41.76667,0.0,50.0,2,0bab3b63-5d31-4de3-85e2-89dc5a3e9bd5,TPP DOOSAN PUMA 240 (TPP) Torn C.N.C,27451f15-090c-42a6-82d6-98b7451db3db,TPU,Torns Puma (Taller COURES),ee76d685-a4cc-47d4-b418-6e33d2ae2921,26a3fecc-120e-49bf-9e77-720d35355e3d,Mecanitzat al Torn Puma Petit,6.0,15.0,1.0,0.0,0.0,29783995-2f56-48be-9664-461e6f0b402a,3.0,2025-10-13 12:55:00.000,2025-10-02 00:00:00.000,00013c31-3895-4370-bb3e-4255371e8d8f


Cuántas máquinas y cuántas fases tiene cada familia:

In [131]:
resumen_familia = df_FASE_final.groupby("NombreGrupo").agg(
    n_fases=("SmFaseUid", "nunique"),
    n_maquinas=("MaquinaDominante", "nunique"),
).sort_values("n_fases", ascending=False)

print(resumen_familia)
print("")
print(f"Total grupos de máquinas: {len(resumen_familia)}")

                                          n_fases  n_maquinas
NombreGrupo                                                  
Torns Puma (Taller COURES)                   4198           2
Màquines de Taladrat Profund                 3656           2
Centres de Mecanitzat Vertical COURES        2719           4
Torns Control Numèric                        1653           4
Centres de Mecanitzat Horitzontal COURES     1298           3
Centre de Mecanitzat 5 Eixos petites          577           4
Centres de Mecanitzat Vertical ALUMINIS       523           3
Premses Conformats                            330           2
Màquines Secció Flexibles                     176           4

Total grupos de máquinas: 9


In [132]:
display(resumen_familia)

,n_fases,n_maquinas
NombreGrupo,,
Torns Puma (Taller COURES),4198,2
Màquines de Taladrat Profund,3656,2
Centres de Mecanitzat Vertical COURES,2719,4
Torns Control Numèric,1653,4
Centres de Mecanitzat Horitzontal COURES,1298,3
Centre de Mecanitzat 5 Eixos petites,577,4
Centres de Mecanitzat Vertical ALUMINIS,523,3
Premses Conformats,330,2
Màquines Secció Flexibles,176,4


Toca observar si las imputaciones se reparten entre las máquinas de la família.

In [133]:
familias_analizables = set(df_FASE_final["NombreGrupo"])

for familia in familias_analizables:
    sub = df_FASE_final[df_FASE_final["NombreGrupo"] == familia]
    reparto = sub["MaquinaDominante"].value_counts(normalize=True)
    print(f"{familia[:45]:45s} | fases: {len(sub):5d} | máq. principal: {reparto.iloc[0]:.0%} | nº máq. usadas: {(reparto > 0.001).sum()}")

Premses Conformats                            | fases:   330 | máq. principal: 57% | nº máq. usadas: 2
Torns Control Numèric                         | fases:  1653 | máq. principal: 34% | nº máq. usadas: 4
Torns Puma (Taller COURES)                    | fases:  4198 | máq. principal: 79% | nº máq. usadas: 2
Centres de Mecanitzat Horitzontal COURES      | fases:  1298 | máq. principal: 72% | nº máq. usadas: 3
Centre de Mecanitzat 5 Eixos petites          | fases:   577 | máq. principal: 81% | nº máq. usadas: 4
Centres de Mecanitzat Vertical COURES         | fases:  2719 | máq. principal: 50% | nº máq. usadas: 4
Màquines Secció Flexibles                     | fases:   176 | máq. principal: 98% | nº máq. usadas: 4
Centres de Mecanitzat Vertical ALUMINIS       | fases:   523 | máq. principal: 66% | nº máq. usadas: 3
Màquines de Taladrat Profund                  | fases:  3656 | máq. principal: 88% | nº máq. usadas: 2


## Decisión de Viabilidad 
Tras todas las limpiezas que he realizado tengo ya un bloque sólido de famílias, tengo 9 familias con elección real y volumen.

Este es el resultado que cierra la viabilidad, y la conclusión es positiva.

- Torns Control Numèric — 34% principal, 4 máquinas usadas. Excelente. El trabajo está muy repartido entre las 4 — justo lo que valida la fusión realizada, y el mejor ejemplo de elección real. 1.672 fases.
- Centres Vertical COURES — 49% principal, 4 máquinas usadas. Reparto sano sobre 2.750 fases. Familia estrella.
- Premses Conformats — 51% principal, 2 máquinas usadas. Reparto equilibrado entre las dos máquinas activas, aunque con volumen moderado (367 fases).
- Centres Vertical ALUMINIS — 66% principal, 3 máquinas usadas. Correcto, 552 fases.
- Centres Horizontal COURES — 72% principal, 3 máquinas usadas. Aceptable, 1.330 fases.

Este es el núcleo sólido: hay decisión de asignación efectiva y reparto real del trabajo entre máquinas.

- Torns Puma — 79% principal, 2 máquinas usadas. La principal pesa, pero el 21% restante sobre 4.219 fases deja un volumen amplio en la segunda máquina. Utilizable, con matiz.
- Centre 5 Eixos petites — 80% principal, 4 máquinas usadas. La principal domina, pero hay reparto entre las cuatro. 603 fases. Utilizable.
- Màquines de Taladrat Profund — 88% principal, 2 máquinas usadas. Una máquina hace casi todo, pero con 3.702 fases el 12% restante sigue siendo un volumen relevante en la segunda.

Estas tres son frontera: en principio cuento con ellas, pero debo estar dispuesto a descartarlas según cómo evolucione el análisis.

- Màquines Secció Flexibles — 95% principal, y solo 188 fases. Aquí sí: una máquina acapara casi todo y el volumen es bajo. La elección es prácticamente ficticia. La excluiría del análisis de idoneidad, o la mencionaría explícitamente como familia sin elección efectiva. No aporta.

## Conclusión
De las familias productivas con múltiples máquinas, la mayoría presentan un reparto efectivo del trabajo entre sus máquinas, lo que confirma que la asignación fase-máquina constituye una decisión real y no determinista en el taller. Esto valida empíricamente la premisa central del TFM: existe un problema de asignación con elección genuina que un modelo de idoneidad puede aprender a optimizar.

NOTA PERSONAL:
puede que haya criterios legítimos (tamaño de pieza, disponibilidad, precisión requerida) que hacen que cierta máquina sea la natural para la mayoría de fases, reservando las otras para casos concretos. Eso también es idoneidad, y de hecho es lo interesante que tu modelo podría capturar. Así que no descartes las de principal alta a la ligera; el matiz es "poca elección" en Secció Flexibles (por volumen + concentración), no en las demás.

## Justificación
Análisis de viabilidad: existencia de una decisión de asignación real

Antes de abordar el modelado, resulta imprescindible verificar una premisa sobre la que se sostiene todo el trabajo: que la asignación de una fase a una máquina constituye efectivamente una decisión con alternativas, y no un proceso determinista en el que cada fase se ejecuta necesariamente en un único recurso posible. Si no existiera tal elección, no habría idoneidad que evaluar ni asignación que optimizar, y el problema carecería de sentido.

Para comprobarlo, se ha analizado el conjunto de familias de máquina productivas resultantes del proceso de depuración, cada una de las cuales agrupa máquinas funcionalmente equivalentes entre las que el taller puede repartir el trabajo. Se ha exigido, como condición necesaria, que una familia disponga de al menos dos máquinas; las familias constituidas por una única máquina se excluyeron de este análisis por no ofrecer alternativa alguna de asignación. Sobre las familias restantes, se ha examinado no solo cuántas máquinas podrían ejecutar sus fases, sino en qué medida el trabajo se reparte realmente entre ellas, calculando la proporción de fases que concentra la máquina más utilizada de cada familia.

Los resultados confirman la existencia de una decisión de asignación efectiva. En la mayoría de las familias analizadas, la máquina principal concentra entre el 33 % y el 70 % de las fases, evidenciando que el trabajo se distribuye de forma genuina entre varios recursos y que, por tanto, existe una elección real en la asignación. Casos como el de los tornos de control numérico (33 % en la máquina principal, con las fases repartidas entre cuatro máquinas) o los centros de mecanizado vertical de cobre (43 %) constituyen la evidencia más clara de que la asignación no es determinista. Se observaron asimismo familias con una concentración elevada en una única máquina (superior al 85 %), que, si bien disponen de alternativas, presentan una elección efectiva más limitada; estas se tratan con cautela en el análisis posterior.

En conjunto, este análisis valida empíricamente la premisa central del trabajo: existe un problema de asignación fase-máquina con capacidad de elección real, sobre un volumen de datos suficiente, que un modelo de idoneidad puede aprender a caracterizar y optimizar.

En conjunto, tras el proceso completo de depuración, el análisis se sostiene sobre un total de 21.206 fases distribuidas en las familias de máquina productivas con capacidad de elección, un volumen que proporciona una base de datos amplia y suficiente para el modelado posterior. Este análisis valida empíricamente la premisa central del trabajo: existe un problema de asignación fase-máquina con capacidad de elección real, sobre un volumen de datos suficiente, que un modelo de idoneidad puede aprender a caracterizar y optimizar.

Con esto la sección de viabilidad queda completa y con las cifras reales: partiste de 140.485 imputaciones en crudo y has destilado una población limpia de 21.206 fases analizables, cada una con su máquina, su familia y sus tiempos reales

# Construir el ratio real/teórico:

In [134]:
display(df_FASE_final.sample(5))

,SmFaseUid,MaquinaDominante,smOrdenId,smMaterialId,smFaseNme,smOrdenNme,MinutosReal_efectivo,MinutosPreparacion_efectivo,MinutosProduccion_efectivo,MinutosReal_total,MinutosPreparacion_total,MinutosProduccion_total,dedicacion_media,n_imputaciones,IDMaquina,NombreMaquina,IDGrupo,CodigoGrupo,NombreGrupo,IDMOTask,IDManufacturingOrder,Description,PreparationTime,ExecutionTime,TimeUnit,ProductEntry,MaterialExit,IDUsualTask,Quantity,ManualStartDate,RealEndDate,MESEntityId
5138,4c6ef4fe-326f-4923-ad85-2890f5545d9c,FL01,OF2506628,CFFXA1X0018,FLX,Shunt BTV0353,155.68333,34.15000,121.53333,155.68333,34.15000,121.53333,50.000000,2,9ed1a126-b943-46b8-88f2-441f055ce0ae,MAQ. FLEX.,fdffea54-b090-48e8-95e8-43b152a40380,FLEX,Màquines Secció Flexibles,97055e01-a3fc-43a2-90fc-8fa68fedb3a5,d038be24-b6d6-404c-b401-6a80213ec1d5,Secció Flexibles,38.0,105.0,1.0,0.0,0.0,031d7da6-65d4-40dd-8217-edb3cd033d5b,3.0,2025-07-23 13:35:00.000,NaN,4c6ef4fe-326f-4923-ad85-2890f5545d9c
8125,77db33cd-8e36-418f-8f39-f67e7b7b53e1,CM05,OF2601959,BR3TA2X0045,CHO-CNYPUN,Elektrodenarm C50 Ø50 1159D367973T1,166.95001,166.95000,0.00000,166.95001,166.95000,0.00000,66.666667,3,9835df18-f376-4d2c-a1fc-dea39221366f,CH MAK. a81,05a696bb-8ad5-4b76-af4f-7bd8112857e6,CH,Centres de Mecanitzat Horitzontal COURES,5676290b-bced-42af-acd3-2422002a4c0c,fb3a973e-7478-4d66-be88-d145db6bb3f0,Mecanitzar Canya i Punta al CH,88.0,55.0,1.0,0.0,0.0,24d8fade-bc10-4a5f-9011-ddea24b87f1e,1.0,2026-04-02 21:05:00.000,2026-03-24 00:00:00.000,77db33cd-8e36-418f-8f39-f67e7b7b53e1
17310,ff07d47c-6d24-40fa-9c88-1d78055dbc3e,TP02,OF2510129,PE0XA2X0297,MTP,ALLONGE COUDÉE 103116873,46.45000,46.45000,0.00000,46.45000,46.45000,0.00000,50.000000,2,a045db52-b7d0-4011-b402-e7f63d5e4d3a,MTP IMSA MF800C-HD Màquina Taladres Profunds,7d29197b-e1a1-4c19-bf95-7fa3b96ed500,MTP,Màquines de Taladrat Profund,7ee29956-bee8-49fd-a292-024ef5267423,efae7c55-08c5-442e-88d2-68856f96a232,Màquina de taladrat profund,5.0,9.0,1.0,0.0,0.0,4cfbb761-89c5-49c3-b68a-4a4024d85c8f,3.0,2025-10-30 14:05:00.000,2025-10-17 00:00:00.000,ff07d47c-6d24-40fa-9c88-1d78055dbc3e
12931,bd2f836a-d830-4433-a689-7e2e047d61ab,CM14,OF2507289,BX1XA2X0043,CV-BX,Elektrodenarm 1159D415565T1,162.01667,162.01667,0.00000,162.01667,162.01667,0.00000,50.000000,4,733e8446-eeed-4b61-9125-7d873719dca1,CV DOOSAN DNM-750L,01a3c08e-a022-49c4-9c19-0f1fffd63df7,CV,Centres de Mecanitzat Vertical COURES,8c13e008-e92f-46c5-8d0e-dee6f8737baa,43add2bc-c588-4b80-b749-1d4b678aab11,Mecanitzar Braç de xapa al CV,45.0,60.0,1.0,0.0,0.0,d73a8469-93ee-41c4-8987-29ad596c94e7,1.0,2025-07-25 10:34:00.000,2025-07-29 00:00:00.000,bd2f836a-d830-4433-a689-7e2e047d61ab
9673,8f1e0331-6139-4e22-a040-4adc2d7c497d,CM06,OF2507417,PE1XNCX0166,PE01,Klemmschaft Ø32 H80 90 (ST59D370624T1),104.78333,104.78333,0.00000,104.78333,104.78333,0.00000,50.000000,2,fd6f1a2e-b80d-4713-8155-02da8069c0b2,CV KONDIA HM-2010 CU,01a3c08e-a022-49c4-9c19-0f1fffd63df7,CV,Centres de Mecanitzat Vertical COURES,28cd0f32-162a-463b-b9c4-8189aecb078f,3cbcf1ee-3ebe-4089-be3a-72899c77408d,Mecanitzar Portaelectrodes al CV Kondia,23.0,48.0,1.0,0.0,0.0,e2a92609-a133-4161-93c1-954f0bdeca5b,2.0,2025-08-08 14:44:00.000,2025-08-27 00:00:00.000,8f1e0331-6139-4e22-a040-4adc2d7c497d


Vamos a trabajar con tiempo total, en las imputaciones aveces todo el peso se queda en preparación o producción, sin separlo, esto se debe a una mala disciplina entre los operarios. 

In [135]:
# ¿cuántas fases tienen teórico en una parte pero real 0 en esa misma parte?
prep_teorico_sin_real = ((df_FASE_final["PreparationTime"] > 0) & (df_FASE_final["MinutosPreparacion_efectivo"] == 0)).sum()
prod_teorico_sin_real = ((df_FASE_final["ExecutionTime"] > 0) & (df_FASE_final["MinutosProduccion_efectivo"] == 0)).sum()

print(f"Fases con teórico de preparación pero real de preparación = 0: {prep_teorico_sin_real}")
print(f"Fases con teórico de producción pero real de producción = 0: {prod_teorico_sin_real}")
print(f"Total fases: {len(df_FASE_final)}\n")


Fases con teórico de preparación pero real de preparación = 0: 385
Fases con teórico de producción pero real de producción = 0: 12371
Total fases: 15130



Casi todas las fases tienen teórico en una parte pero real 0 en esa misma parte, significa que la separación preparación/producción en el MES no es fiable: el trabajo se imputa a una casilla u otra sin correspondencia con cómo lo definió el ERP.

In [136]:
# --- Construcción del ratio sobre el tiempo EFECTIVO (corregido por dedicación) ---
df_FASE_final["tiempo_real_efectivo"] = df_FASE_final["MinutosPreparacion_efectivo"] + df_FASE_final["MinutosProduccion_efectivo"]
df_FASE_final["tiempo_teorico_total"] = df_FASE_final["PreparationTime"] + df_FASE_final["ExecutionTime"]

df_FASE_final["ratio"] = df_FASE_final["tiempo_real_efectivo"] / df_FASE_final["tiempo_teorico_total"]

# --- Verificaciones ---
# ningún teórico total en 0 (rompería la división)
print(f"Teórico total = 0: {(df_FASE_final['tiempo_teorico_total'] == 0).sum()}")
# ninguna fase con tiempo real efectivo 0 (ratio = 0, sin señal)
print(f"Real efectivo = 0 (ratio 0): {(df_FASE_final['tiempo_real_efectivo'] == 0).sum()}")

# distribución del ratio
print("\nDistribución del ratio (con tiempo efectivo):")
print(df_FASE_final["ratio"].describe())

display(df_FASE_final.sample(2))

Teórico total = 0: 0
Real efectivo = 0 (ratio 0): 54

Distribución del ratio (con tiempo efectivo):
count    15130.000000
mean         2.168678
std          2.897306
min          0.000000
25%          0.987339
50%          1.500000
75%          2.461219
max        150.616665
Name: ratio, dtype: float64


,SmFaseUid,MaquinaDominante,smOrdenId,smMaterialId,smFaseNme,smOrdenNme,MinutosReal_efectivo,MinutosPreparacion_efectivo,MinutosProduccion_efectivo,MinutosReal_total,MinutosPreparacion_total,MinutosProduccion_total,dedicacion_media,n_imputaciones,IDMaquina,NombreMaquina,IDGrupo,CodigoGrupo,NombreGrupo,IDMOTask,IDManufacturingOrder,Description,PreparationTime,ExecutionTime,TimeUnit,ProductEntry,MaterialExit,IDUsualTask,Quantity,ManualStartDate,RealEndDate,MESEntityId,tiempo_real_efectivo,tiempo_teorico_total,ratio
5070,4b81d8d7-aef6-4981-9f2a-3a11d4e39e28,CM06,OF2509104,PE2XA3X0041,PE02,Allonge 7091901394001,181.158335,362.316665,0.000000,362.31667,724.63333,0.00000,25.0,2,fd6f1a2e-b80d-4713-8155-02da8069c0b2,CV KONDIA HM-2010 CU,01a3c08e-a022-49c4-9c19-0f1fffd63df7,CV,Centres de Mecanitzat Vertical COURES,cfa91f52-52a2-4461-b326-ab6ebaf6c360,aa3e313c-eaeb-4f6c-a659-8465aebfbae3,Mecanitzar Portaelectrodes al CVP,50.0,104.0,1.0,0.0,0.0,eea2b66d-b07b-47f6-b276-e55da321def3,2.0,2025-09-30 13:21:00.000,2025-09-29 00:00:00.000,4b81d8d7-aef6-4981-9f2a-3a11d4e39e28,362.316665,154.0,2.352706
13300,c305115d-c11c-4089-9efc-2083819cbfc8,TN08,OF2506691,PA0XA2X0057,TCN,Allonge N001362701(IM01421392),24.791665,0.100000,49.483335,49.58333,0.20000,98.96667,25.0,2,bf44bec2-aa1e-4002-8da5-922a1001645d,HYUNDAI-KIA TURN H15 Torn C.N.C,6977175d-a2da-4925-a70f-fd0fcc4a2960,TCN,Torns Control Numèric,203d7b52-29ff-47c2-adff-2b04304c3986,8dfb5d52-77b5-45eb-bd7e-2c9376e603de,Mecanització al H15 FORADAR Ø10,28.0,0.0,1.0,0.0,0.0,64dbdeef-a81c-42f8-83fa-ac6e953af375,1.0,2025-07-15 23:32:00.000,2025-07-14 00:00:00.000,c305115d-c11c-4089-9efc-2083819cbfc8,49.583335,28.0,1.770833


Los tiempos los tengo representados asi:
       
Los efectivos (corregidos con la dedicación) — los que usas para el ratio:     
`MinutosReal_efectivo` — tiempo real total ponderado por dedicación      
`MinutosPreparacion_efectivo` — preparación ponderada     
`MinutosProduccion_efectivo` — producción ponderada     
      
Los brutos (tiempo de reloj sin corregir) — los conservo para trazabilidad/comparación:     
`MinutosReal_total`      
`MinutosPreparacion_total`     
`MinutosProduccion_total`      

> Distinción clave: el bruto es el tiempo de reloj (la máquina estuvo abierta X minutos), el efectivo descuenta el solapamiento (× dedicación/100). El teórico asume dedicación plena, así que solo es comparable con el efectivo. Por eso el ratio usa el efectivo.    

Tiempos TEÓRICOS del ERP (lo que debería tardar):     
`PreparationTime` — teórico de preparación     
`ExecutionTime` — teórico de ejecución/producción     
      
Los tiempos TOTALES que he construido para el ratio:     
`tiempo_real_efectivo` = MinutosPreparacion_efectivo + MinutosProduccion_efectivo       
`tiempo_teorico_total` = PreparationTime + ExecutionTime       

Y la variable objetivo:     
`ratio` = tiempo_real_efectivo / tiempo_teorico_total     

Los 54 con ratio = 0 (real efectivo nulo). Estas fases tienen teórico pero su tiempo real efectivo es 0 — probablemente fases cuyas imputaciones tenían todas dedicación 0, o tiempo real 0. Un ratio de 0 no significa "idoneidad perfecta", significa "sin dato de tiempo real". Hay que descartarlas, mismo argumento que los teóricos 0: sin real medible, no hay idoneidad.

In [137]:
antes = len(df_FASE_final)
df_FASE_final = df_FASE_final[df_FASE_final["tiempo_real_efectivo"] > 0]
print(f"Descartadas (ratio 0): {antes - len(df_FASE_final)}")
print(f"Total imputaciones: {len(df_FASE_final)}")

Descartadas (ratio 0): 54
Total imputaciones: 15076


El máximo de 150.61 es demasiado alto. Un ratio de 242 significa una fase que tardó 242 veces su tiempo teórico — físicamente imposible como "mala idoneidad" real. Es ruido residual (solapamiento que la dedicación no capturó del todo, teóricos minúsculos mal cargados, o algún caso raro). 

In [138]:
# Inspecciono los peores:
#display(df_FASE_final.nlargest(20, "ratio")[["SmFaseUid", "MaquinaDominante", "NombreGrupo", "smFaseNme", "tiempo_real_efectivo", "tiempo_teorico_total", "ratio", "dedicacion_media", "n_imputaciones"]])

Esta inspección me muestra outliers, estas imputaciones teienen unas desviaciones altisimas, la imputación no ha capturado bien el tiempo, recogiendo mucho más del esperado.

Aqui el problema viene que esos tiempos teoricos estan en horas y no en minutos, hay que arreglarlo.

In [139]:
# --- Exploración de ambas colas antes de cortar ---
print("Distribución del ratio:")
print(df_FASE_final["ratio"].describe())
print()

# cola alta: cuántas fases con distintos umbrales
print("=== COLA ALTA (tardó mucho más que el teórico) ===")
for umbral in [5, 8, 10, 15, 20]:
    n = (df_FASE_final["ratio"] >= umbral).sum()
    print(f"  Ratio >= {umbral:>2}: {n:>4} fases ({n/len(df_FASE_final):.1%})")

# cola baja: cuántas fases sospechosamente rápidas
print("\n=== COLA BAJA (tardó mucho menos que el teórico) ===")
for umbral in [0.05, 0.1, 0.2, 0.3]:
    n = (df_FASE_final["ratio"] <= umbral).sum()
    print(f"  Ratio <= {umbral}: {n:>4} fases ({n/len(df_FASE_final):.1%})")

# percentiles para calibrar el corte con criterio estadístico
print("\n=== PERCENTILES ===")
for p in [0.01, 0.02, 0.05, 0.95, 0.98, 0.99]:
    print(f"  P{int(p*100):>2}: {df_FASE_final['ratio'].quantile(p):.3f}")

Distribución del ratio:
count    15076.000000
mean         2.176445
std          2.899577
min          0.000002
25%          0.993101
50%          1.506410
75%          2.468668
max        150.616665
Name: ratio, dtype: float64

=== COLA ALTA (tardó mucho más que el teórico) ===
  Ratio >=  5:  994 fases (6.6%)
  Ratio >=  8:  369 fases (2.4%)
  Ratio >= 10:  223 fases (1.5%)
  Ratio >= 15:   92 fases (0.6%)
  Ratio >= 20:   51 fases (0.3%)

=== COLA BAJA (tardó mucho menos que el teórico) ===
  Ratio <= 0.05:  175 fases (1.2%)
  Ratio <= 0.1:  229 fases (1.5%)
  Ratio <= 0.2:  332 fases (2.2%)
  Ratio <= 0.3:  463 fases (3.1%)

=== PERCENTILES ===
  P 1: 0.039
  P 2: 0.169
  P 5: 0.439
  P95: 5.656
  P98: 8.571
  P99: 12.145


En base al `df_FASE_final['ratio']` viendo los desvios, voy a tomar un corte [0.2, 8.5]
- Corte por arriba: un ratio ≥10 (tardó 10 veces lo previsto) es muy difícil como rendimiento real de máquina (para nuestra población,imposible), coincide bien con estar cerca del percentil 99%.

- Cola baja: ≤0.1 son 229 fases (1.5%). Un ratio de 0.1 significa que la fase se hizo en el 10% del tiempo teórico — sospechoso (teórico inflado o real incompleto). 0.1 es un corte razonable por abajo, simétrico en volumen con el de arriba (229 vs 223, casualmente parecidos).

Entonces estas fases debido a su mala disciplina en la imputación las trataré como outliers. 

Como digo Outliers que son ruido (MES abierto, errores de registro, teóricos mal cargados) → hay que eliminarlos, contaminan el aprendizaje.

In [140]:
UMBRAL_ALTO = 8.5
UMBRAL_BAJO = 0.2

antes = len(df_FASE_final)
alta = (df_FASE_final["ratio"] >= UMBRAL_ALTO).sum()
baja = (df_FASE_final["ratio"] <= UMBRAL_BAJO).sum()

df_FASE_final = df_FASE_final[(df_FASE_final["ratio"] > UMBRAL_BAJO) & (df_FASE_final["ratio"] < UMBRAL_ALTO)]

print(f"Descartadas cola alta (>= {UMBRAL_ALTO}): {alta}")
print(f"Descartadas cola baja (<= {UMBRAL_BAJO}): {baja}")
print(f"Total: {antes} → {len(df_FASE_final)}")

Descartadas cola alta (>= 8.5): 308
Descartadas cola baja (<= 0.2): 332
Total: 15076 → 14436


> Los cortes son asimétricos en interpretación, aunque parezcan simétricos en número. 
- El de arriba (≥9) quita "tardó absurdamente mucho", casi seguro tiempo muerto. 
- El de abajo (≤0.2) quita "tardó absurdamente poco", casi seguro teórico mal cargado o real incompleto. 
>Son dos problemas de datos distintos que dan la misma señal (ratio implausible), y ambos justifican el descarte.

In [141]:
print("Distribución del ratio:")
print(df_FASE_final["ratio"].describe())

Distribución del ratio:
count    14436.000000
mean         1.944248
std          1.411691
min          0.201455
25%          1.017204
50%          1.509392
75%          2.403604
max          8.498575
Name: ratio, dtype: float64


Limpio:

In [148]:
df_FASE_final2 = df_FASE_final.drop(columns=["IDMOTask", "IDManufacturingOrder", "IDMaquina", "IDGrupo", "Description", "MinutosPreparacion_total", "MinutosProduccion_total", "MinutosReal_total", "MESEntityId", "ExecutionTime", "PreparationTime", "ManualStartDate", "RealEndDate"])
df_FASE_final2 = df_FASE_final2[["SmFaseUid", "MaquinaDominante", "NombreMaquina", "CodigoGrupo", "NombreGrupo", "smOrdenId", "smOrdenNme", "smFaseNme", "IDUsualTask", "smMaterialId", "Quantity", "tiempo_real_efectivo", "tiempo_teorico_total", "ratio", "ProductEntry", "MaterialExit", "n_imputaciones"]]
display(df_FASE_final2.sample(2))
print(df_FASE_final2[["Quantity", "tiempo_real_efectivo", "tiempo_teorico_total", "ratio"]].describe())

df_FASE_final2_id = df_FASE_final[["SmFaseUid", "IDMOTask", "IDManufacturingOrder", "IDMaquina", "IDGrupo"]]\
.rename(columns={
        "IDMOTask": "ID_Task_ERP",
        "IDManufacturingOrder": "ID_OF_ERP",
        })
display(df_FASE_final2_id.sample(2))

,SmFaseUid,MaquinaDominante,NombreMaquina,CodigoGrupo,NombreGrupo,smOrdenId,smOrdenNme,smFaseNme,IDUsualTask,smMaterialId,Quantity,tiempo_real_efectivo,tiempo_teorico_total,ratio,ProductEntry,MaterialExit,n_imputaciones
6820,64f016c1-d69b-4922-812b-dccda874a708,CM05,CH MAK. a81,CH,Centres de Mecanitzat Horitzontal COURES,OF2600564,Electrode arm Ø60 1159D363095,CHO-CNYPUN,24d8fade-bc10-4a5f-9011-ddea24b87f1e,BR0TA2X0001,1.0,109.80000,110.0,0.998182,0.0,0.0,2
14441,d413968f-9174-4408-849d-a647d2470d2e,CM05,CH MAK. a81,CH,Centres de Mecanitzat Horitzontal COURES,OF2601599,COOPER PART Arm B2PN54CA1655,CHO-PUN,987b3954-076b-4c65-9909-86c878839853,BR2XA3X0208,2.0,120.36667,116.0,1.037644,0.0,0.0,2


           Quantity  tiempo_real_efectivo  tiempo_teorico_total         ratio
count  14436.000000          14436.000000          14436.000000  14436.000000
mean       5.191535            108.290621             75.579800      1.944248
std       27.247462            105.511610             94.616208      1.411691
min        1.000000              1.704286              5.000000      0.201455
25%        1.000000             34.983330             19.000000      1.017204
50%        2.000000             72.808330             46.500000      1.509392
75%        3.000000            143.895830             95.000000      2.403604
max     1500.000000            941.583340           1560.000000      8.498575


,SmFaseUid,ID_Task_ERP,ID_OF_ERP,IDMaquina,IDGrupo
7483,6e641e5e-a8a8-4b9d-ba16-450f720389b7,93596afb-2993-42ed-90f7-3cabb4e1ce7c,1e0ca9b6-fde2-413a-84e3-1ecafa62ecd3,a045db52-b7d0-4011-b402-e7f63d5e4d3a,7d29197b-e1a1-4c19-bf95-7fa3b96ed500
4357,40de3d52-3151-4061-86de-36344741d663,05337196-e46a-41e9-8e2c-bed5136b6506,40a2feca-1eea-47c3-8fa2-4ce18ef85e16,fd6f1a2e-b80d-4713-8155-02da8069c0b2,01a3c08e-a022-49c4-9c19-0f1fffd63df7


In [144]:
#Inspecciono los peores tras la limpieza
#display(df_FASE_final.nlargest(20, "ratio")[["SmFaseUid", "MaquinaDominante", "NombreGrupo", "smFaseNme", "tiempo_real_efectivo", "tiempo_teorico_total", "ratio", "dedicacion_media", "n_imputaciones"]])

In [149]:
df_FASE_final2.to_parquet("res//procesed/df_FASE_final2.parquet")

df_FASE_final2_id.to_parquet("res//procesed/df_FASE_final2_id.parquet")